# Juntar todos los parquets con las clases y el point

- centros_comerciales    840
- mercados               329
- sitios_his             163 (sitios históricos)
- top_100_spots          150
- centros_culturales      93
- sitios_arq              42 (sitios arquelógicos)
- deportivos              32

In [3]:
# -*- coding: utf-8 -*-
from pathlib import Path
import re
from unidecode import unidecode
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ========= Config =========
raw_folder = Path("Raw")
final_folder = Path("Final")
TARGET_EPSG = 6732
OUT = final_folder / "tourism.parquet"

# ========= Helpers =========
def norm(s: str) -> str:
    s = unidecode(str(s)).lower().strip()
    s = re.sub(r"[^\w\s]", "_", s)
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"_+", "_", s)
    return s.strip("_")

def read_geoparquet_fix(path: Path) -> gpd.GeoDataFrame:
    """Lee un parquet, asegura CRS, descarta Z y solo deja POINTs."""
    g = gpd.read_parquet(path)

    # Geometría activa
    try:
        cur = g.geometry.name
    except Exception:
        candidates = [c for c in g.columns if getattr(g[c].dtype, "name", "") == "geometry"]
        if not candidates:
            raise ValueError(f"No se detectó geometría en {path}")
        g = g.set_geometry(candidates[0])
        cur = candidates[0]

    if cur != "geom":
        if "geom" in g.columns and getattr(g["geom"].dtype, "name", "") != "geometry":
            g = g.drop(columns=["geom"])
        g = g.rename_geometry("geom")

    # Reproyección
    if g.crs is None:
        g = g.set_crs(TARGET_EPSG, allow_override=True)
    elif g.crs.to_epsg() != TARGET_EPSG:
        g = g.to_crs(TARGET_EPSG)

    # Forzar solo POINT 2D
    g = g[g.geom_type == "Point"].copy()
    g["geom"] = g["geom"].apply(lambda p: Point(p.x, p.y) if p is not None else None)

    return g

# ========= Build master =========
frames = []
log = []

for pq in sorted(raw_folder.glob("*.parquet")):
    g = read_geoparquet_fix(pq)

    # Si el parquet ya trae "clase"
    if "clase" in g.columns:
        tmp = g[["clase", "geom"]].copy()
        tmp["clase"] = tmp["clase"].map(norm)
    else:
        clase = norm(pq.stem)  # usar nombre del archivo como clase
        tmp = g[["geom"]].copy()
        tmp["clase"] = clase

    tmp = tmp[["clase", "geom"]]
    frames.append(tmp)
    log.append((pq.name, len(tmp)))

if not frames:
    raise RuntimeError("No se encontraron parquets en la carpeta raw/.")

final = pd.concat(frames, ignore_index=True)
final.insert(0, "id", range(len(final)))
final = gpd.GeoDataFrame(final, geometry="geom", crs=f"EPSG:{TARGET_EPSG}")

# Guardar
final_folder.mkdir(exist_ok=True, parents=True)
final.to_parquet(OUT)
print(f"✅ Guardado {len(final)} registros en {OUT}")

# Resumen
summary = pd.DataFrame(log, columns=["parquet", "rows"])
print("\nResumen por archivo:")
print(summary.to_string(index=False))

print("\nFrecuencia por clase:")
print(final["clase"].value_counts())


✅ Guardado 1649 registros en Final\tourism.parquet

Resumen por archivo:
                    parquet  rows
centros_comerciales.parquet   840
     cultura_points.parquet   298
         deportivos.parquet    32
           mercados.parquet   329
       top100_spots.parquet   150

Frecuencia por clase:
clase
centros_comerciales    840
mercados               329
sitios_his             163
top_100_spots          150
centros_culturales      93
sitios_arq              42
deportivos              32
Name: count, dtype: int64


In [4]:
check = gpd.read_parquet(final_folder/"tourism.parquet")
check.head()

,id,clase,geom
0,0,centros_comerciales,POINT (-1394798.773 27756868.958)
1,1,centros_comerciales,POINT (-1394552.091 27757288.164)
2,2,centros_comerciales,POINT (-1394617.953 27756663.147)
3,3,centros_comerciales,POINT (-1394797.581 27756271.417)
4,4,centros_comerciales,POINT (-1394681.105 27757630.065)


In [5]:
# Detectar si alguna geometría tiene Z
check["has_z"] = check["geom"].apply(lambda g: g.has_z if g is not None else False)

print(check["has_z"].value_counts())


has_z
False    1649
Name: count, dtype: int64
